In [4]:
import subprocess
subprocess.run(["pip", "install", "yfinance", "kafka-python"], capture_output=False)

CompletedProcess(args=['pip', 'install', 'yfinance', 'kafka-python'], returncode=0)

In [5]:
import yfinance as yf
import json
import time
from datetime import datetime
from kafka import KafkaProducer

In [6]:
producer = KafkaProducer(
    bootstrap_servers=["kafka:29092"],
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8")
)

print("✅ Connected to Kafka !")

✅ Connected to Kafka !


In [7]:
def fetch_ticker(symbol):
    t = yf.Ticker(symbol)
    info = t.fast_info
    return {
        "symbol":     symbol,
        "timestamp":  datetime.utcnow().isoformat(),
        "price":      round(float(info.last_price), 4) if info.last_price else None,
        "volume":     int(info.last_volume) if info.last_volume else None,
        "market_cap": float(info.market_cap) if info.market_cap else None,
    }

# Test it on one stock
print(fetch_ticker("AAPL"))

{'symbol': 'AAPL', 'timestamp': '2026-04-26T22:56:21.199116', 'price': 271.06, 'volume': 38124500, 'market_cap': 3979469772557.373}


In [8]:
TOPIC = "financial-data"
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

for ticker in TICKERS:
    record = fetch_ticker(ticker)
    producer.send(TOPIC, key=ticker, value=record)
    print(f"Sent → {record['symbol']} | ${record['price']}")

producer.flush() #producer.flush() means "make sure everything was actually sent before moving on"
print("\n✅ All sent to Kafka!")

Sent → AAPL | $271.06
Sent → MSFT | $424.62
Sent → GOOGL | $344.4
Sent → AMZN | $263.99
Sent → TSLA | $376.3

✅ All sent to Kafka!
